In [1]:
!pip install --upgrade scikeras
!pip install tensorflow
!pip install pandas
!pip install numpy
!pip install scikit-learn==1.4.2
!pip install tensorboard
!pip install matplotlib
!pip install streamlit
!pip install keras

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
data = pd.read_csv("/content/Churn_Modelling.csv")

In [4]:
data.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
## preprocessing : drop irrelevant features
data= data.drop(["RowNumber","CustomerId","Surname"],axis=1)

In [6]:
## Encoding Categorical Variable
label_encoder=LabelEncoder()

In [7]:
data["Gender"]=label_encoder.fit_transform(data['Gender'])

In [8]:
from sklearn.preprocessing import OneHotEncoder

In [9]:
onehot_encoder=OneHotEncoder()
data_encoder=onehot_encoder.fit_transform(data[['Geography']])

In [10]:
onehot_encoder.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [11]:
data_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [12]:
data_encoder_df= pd.DataFrame(data_encoder.toarray(),columns=onehot_encoder.get_feature_names_out(['Geography']))

In [13]:
data=pd.concat([data.drop('Geography',axis=1),data_encoder_df],axis=1)

In [14]:
import pickle

In [15]:
##save the encoders
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder,file)


with open('data_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder,file)

In [16]:
y= data['Exited']
X= data.drop('Exited',axis=1)

In [17]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
scaler=StandardScaler()

In [19]:
X_train =scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [20]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [26]:
## Define a function to create the model and try different parameters(KerasClassifier)
def create_model(neurons=32, layers=1, input_shape=(X_train.shape[1],)):
    model = Sequential()
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(Dense(neurons, activation='relu'))
    for i in range(layers):
      model.add(Dense(neurons,activation='relu'))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [27]:
#create a Keras Classifier
model = KerasClassifier(model=create_model, epochs=50, batch_size=10, verbose=0)

In [28]:
#Define the grid search parameters
param_grid= {
    "model__neurons": [16, 32, 64,],
    "model__layers": [1,2],
}

In [29]:
from sklearn.model_selection import GridSearchCV

In [30]:
## perform grid search
grid = GridSearchCV(estimator=model,param_grid=param_grid,cv=3)
grid_result=grid.fit(X_train,y_train)

In [31]:
grid_best_params=grid_result.best_params_
grid_best_score=grid_result.best_score_
print(f"Best Parameters: {grid_best_params}")
print(f"Best Score: {grid_best_score}")

Best Parameters: {'model__layers': 1, 'model__neurons': 16}
Best Score: 0.8523744171888491
